# Day 5: API Fundamentals and FastAPI Core
**REST, HTTP, FastAPI routing, Pydantic models, CRUD and reviewing a Codex endpoint**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi uvicorn httpx requests pydantic

# ===== Option B: VS Code / local (run once in the terminal, not here) =====
# python -m venv .venv
# .venv\Scripts\activate        (Windows)   |   source .venv/bin/activate   (macOS/Linux)
# pip install fastapi uvicorn httpx requests pydantic
import fastapi, pydantic
print("FastAPI", fastapi.__version__, "| Pydantic", pydantic.__version__)

## 2. HTTP from the client side
This cell calls a free public test API. It needs internet access.

In [ ]:
import requests
r = requests.get("https://jsonplaceholder.typicode.com/todos/1", timeout=10)
print("Status:", r.status_code)
print("Content-Type:", r.headers["Content-Type"])
print("Body:", r.json())

r = requests.post("https://jsonplaceholder.typicode.com/posts", json={"title": "Hi", "body": "Day 5", "userId": 1}, timeout=10)
print(r.status_code, r.json())

## 3. Your first FastAPI app
`%%writefile` saves the cell as `main.py`. The same file works in VS Code.

In [ ]:
%%writefile main.py
from fastapi import APIRouter, FastAPI, HTTPException, Query

app = FastAPI(title="Quotes API", version="1.0")
router = APIRouter(prefix="/quotes", tags=["Quotes"])

quotes = {
    1: {"text": "Small steps every day.", "author": "Unknown"},
    2: {"text": "Fall seven times, stand up eight.", "author": "Japanese proverb"},
}

@app.get("/")
def home():
    return {"message": "Welcome to the Quotes API"}

@app.get("/health")
def health():
    return {"status": "ok"}

@router.get("")
def list_quotes(author: str | None = None, limit: int = Query(10, ge=1, le=50)):
    items = [{"id": k, **v} for k, v in quotes.items() if author is None or v["author"] == author]
    return items[:limit]

@router.get("/{qid}")
def get_quote(qid: int):
    if qid not in quotes:
        raise HTTPException(status_code=404, detail="Quote not found")
    return {"id": qid, **quotes[qid]}

@router.post("", status_code=201)
def create_quote(text: str, author: str = "Unknown"):
    qid = max(quotes, default=0) + 1
    quotes[qid] = {"text": text, "author": author}
    return {"id": qid, **quotes[qid]}

@router.delete("/{qid}", status_code=204)
def delete_quote(qid: int):
    if qid not in quotes:
        raise HTTPException(status_code=404, detail="Quote not found")
    del quotes[qid]

app.include_router(router)

## 4. Call the API
**Option A (Colab):** `TestClient` calls the app directly, no server needed.

**Option B (VS Code):** run `uvicorn main:app --reload` in the terminal and open http://127.0.0.1:8000/docs

In [ ]:
# ===== Option A: Google Colab (active) =====
import importlib, main
importlib.reload(main)
from fastapi.testclient import TestClient
client = TestClient(main.app)

# ===== Option B: VS Code with uvicorn running (uncomment to use) =====
# import requests
# class _Client:
#     base = "http://127.0.0.1:8000"
#     def get(self, p, **k): return requests.get(self.base + p, **k)
#     def post(self, p, **k): return requests.post(self.base + p, **k)
#     def delete(self, p, **k): return requests.delete(self.base + p, **k)
# client = _Client()

print(client.get("/").json())
print(client.get("/quotes/2").json())
print(client.get("/quotes", params={"author": "Unknown"}).json())

In [ ]:
r = client.post("/quotes", params={"text": "Keep learning", "author": "Ahmed"})
print("POST:", r.status_code, r.json())
print("Missing:", client.get("/quotes/99").status_code, client.get("/quotes/99").json())
print("Wrong type:", client.get("/quotes/abc").status_code)
print("Limit too big:", client.get("/quotes", params={"limit": 500}).status_code)
print("DELETE:", client.delete("/quotes/1").status_code)

## 5. Status code check
If nothing prints except the final message, every rule holds.

In [ ]:
assert client.get("/health").status_code == 200
assert client.post("/quotes", params={"text": "Hi"}).status_code == 201
assert client.get("/quotes/999").status_code == 404
assert client.get("/quotes/abc").status_code == 422
print("All status code checks passed")

### TODO 1: Add `GET /quotes/random` and a test for it. Tip: put it above `/quotes/{qid}` so the path matches first.

In [ ]:
# TODO: write your code here

## 6. Pydantic on its own

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class BookCreate(BaseModel):
    title: str = Field(min_length=1, examples=["Ikigai"])
    author: str
    pages: int = Field(gt=0)
    year: int = Field(ge=1450, le=2026)

ok = BookCreate(title="Ikigai", author="Garcia", pages=208, year=2016)
print(ok.model_dump())

try:
    BookCreate(title="", author="X", pages=-5, year=3000)
except ValidationError as e:
    for err in e.errors():
        print(err["loc"], "->", err["msg"])

## 7. A Books API with a router, models and full CRUD

In [ ]:
%%writefile books_api.py
from fastapi import APIRouter, FastAPI, HTTPException, Query
from pydantic import BaseModel, Field

class BookCreate(BaseModel):
    title: str = Field(min_length=1)
    author: str
    pages: int = Field(gt=0)

class BookUpdate(BaseModel):
    title: str | None = Field(default=None, min_length=1)
    author: str | None = None
    pages: int | None = Field(default=None, gt=0)

class Book(BookCreate):
    id: int

books: dict[int, Book] = {}
router = APIRouter(prefix="/books", tags=["Books"])

@router.get("", response_model=list[Book])
def list_books(author: str | None = None, limit: int = Query(10, ge=1, le=100)):
    result = [b for b in books.values() if author is None or b.author == author]
    return result[:limit]

@router.get("/{book_id}", response_model=Book)
def get_book(book_id: int):
    if book_id not in books:
        raise HTTPException(404, "Book not found")
    return books[book_id]

@router.post("", response_model=Book, status_code=201)
def create_book(data: BookCreate):
    new_id = max(books, default=0) + 1
    books[new_id] = Book(id=new_id, **data.model_dump())
    return books[new_id]

@router.patch("/{book_id}", response_model=Book)
def update_book(book_id: int, data: BookUpdate):
    if book_id not in books:
        raise HTTPException(404, "Book not found")
    changes = data.model_dump(exclude_unset=True)
    books[book_id] = books[book_id].model_copy(update=changes)
    return books[book_id]

@router.delete("/{book_id}", status_code=204)
def delete_book(book_id: int):
    if books.pop(book_id, None) is None:
        raise HTTPException(404, "Book not found")

app = FastAPI(title="Books API")
app.include_router(router)

In [ ]:
import importlib, books_api
importlib.reload(books_api)
bc = TestClient(books_api.app)

r = bc.post("/books", json={"title": "Ikigai", "author": "Garcia", "pages": 208})
print(r.status_code, r.json())
bc.post("/books", json={"title": "Deep Work", "author": "Newport", "pages": 296})
print("PATCH:", bc.patch("/books/1", json={"pages": 210}).json())
print("Bad data:", bc.post("/books", json={"title": "", "author": "X", "pages": 0}).status_code)
print("Missing:", bc.get("/books/99").status_code)
print("Delete:", bc.delete("/books/2").status_code)
print("Left:", [b["title"] for b in bc.get("/books").json()])

### TODO 2: Add a `year` field (1450 to 2026) to the models and a query filter `min_pages` to the list endpoint.

In [ ]:
# TODO: write your code here

## 8. NEW (v4): Generate an endpoint with Codex, then review and test it
Below is the kind of code an AI assistant often produces for the prompt "add a PATCH endpoint for tickets". It runs. Read it before you run the tests: can you spot the bug?

In VS Code, try it for real: open the OpenAI Codex panel, paste the prompt from the deck, and review the diff.

In [ ]:
%%writefile tickets_api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

class TicketCreate(BaseModel):
    title: str = Field(min_length=3)
    priority: str = Field(pattern="^(low|medium|high)$")
    status: str = "open"

class TicketUpdate(BaseModel):
    title: str | None = None
    priority: str | None = Field(default=None, pattern="^(low|medium|high)$")
    status: str | None = None

class Ticket(TicketCreate):
    id: int

tickets: dict[int, Ticket] = {1: Ticket(id=1, title="No signal at home", priority="high")}
app = FastAPI(title="SmartDesk tickets")

# ---- generated by an AI assistant, not yet reviewed ----
@app.patch("/tickets/{tid}")
def update_ticket(tid: int, data: TicketUpdate):
    ticket = tickets.get(tid)
    changes = data.model_dump()
    tickets[tid] = ticket.model_copy(update=changes)
    return tickets[tid]

In [ ]:
import importlib, tickets_api
importlib.reload(tickets_api)
tc = TestClient(tickets_api.app, raise_server_exceptions=False)

def review_tests(c):
    results = {}
    r = c.patch("/tickets/1", json={"status": "closed"})
    results["only status changes"] = r.status_code == 200 and r.json()["title"] == "No signal at home"
    results["unknown id gives 404"] = c.patch("/tickets/99", json={"status": "closed"}).status_code == 404
    results["bad priority gives 422"] = c.patch("/tickets/1", json={"priority": "urgent"}).status_code == 422
    for name, ok in results.items():
        print("PASS" if ok else "FAIL", name)
    return all(results.values())

print("All passed:", review_tests(tc))

Two tests fail. `model_dump()` sends `None` for every field the client did not send, so the title is wiped. And a missing ticket crashes with a 500 instead of a clean 404. Here is the reviewed version.

In [ ]:
%%writefile tickets_reviewed.py
from fastapi import FastAPI, HTTPException
from tickets_api import Ticket, TicketUpdate

tickets: dict[int, Ticket] = {1: Ticket(id=1, title="No signal at home", priority="high")}
app = FastAPI(title="SmartDesk tickets (reviewed)")

# ---- reviewed: 404 for unknown ids, only sent fields change, typed response ----
@app.patch("/tickets/{tid}", response_model=Ticket)
def update_ticket(tid: int, data: TicketUpdate):
    if tid not in tickets:
        raise HTTPException(404, "Ticket not found")
    changes = data.model_dump(exclude_unset=True)
    tickets[tid] = tickets[tid].model_copy(update=changes)
    return tickets[tid]

In [ ]:
import tickets_reviewed
importlib.reload(tickets_reviewed)
print("All passed:", review_tests(TestClient(tickets_reviewed.app)))

**Review note template** (paste into your pull request):

* Prompt used: ...
* What I checked: contract, validation, errors, side effects, tests
* What I changed: ...
* Tests added: ...
* Tokens used today (from the Codex usage view): ...

### TODO 3 (SmartDesk step 4): add `POST /tickets` (201) and `DELETE /tickets/{tid}` (204, 404) with tests.

In [ ]:
# TODO: write your code here

## 9. Optional: run a real server inside Colab
This starts uvicorn in the background and calls it over HTTP.

In [ ]:
# ===== Option A: Google Colab =====
import subprocess, time, requests
server = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"])
time.sleep(3)
print(requests.get("http://127.0.0.1:8000/health").json())
server.terminate()

# ===== Option B: VS Code =====
# Just run in the terminal: uvicorn main:app --reload

---
**Yoi shuumatsu wo (良い週末を): *Have a great weekend***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)